# 応用編13

V3.7新機能:秘密分散機能

このノートブックでは、クライアントAPI・バージョン3.7から利用できる秘密分散機能について、使い方を説明します。 


## 秘密分散機能の概要

秘密分散機能は、元データを複数のシェアに秘密分散して異なるピアに格納し、そのうちのいくつかのシェアを集めるだけで、元データを復元できる機能です。  
基本のアルゴリズムとして、Shamir Secret Sharingを使用しています。  
分散するピアの総数をN、復元に必要なピアの数をTとしたとき、
T-1台のピアのシェアを集めただけでは元データを復元できず、  
T台のピアから正常なシェアを集めれば元データが復元できることが保証されます。  
この秘密分散機能はクライアント側のAPIライブラリで提供される機能です。  
ブロックチェーンのストレージ機能を内部で利用しています。  

## 準備

設定やライブラリを読み込みます。  

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { chainID, domain, cnfstr } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, createObjectF } = await import('../lib/notebook-util.mjs');

ピア構成情報(cnfstr)から、ピア構成情報オブジェクト(peerscnf)を得ます。

In [2]:
var peerscnf = await api.loadPeersCnf(cnfstr);

ブロックチェーンに問い合わせ、peerscnfを最新に更新します。

In [3]:
var peerscnf = await rpc.updatePeersCnf(peerscnf);

秘密分散に使用するピアの一覧を設定します。  
この例では、peerscnfから取得します。  

In [4]:
var pid_sockets = [];
for (var [pid, { url }] of peerscnf.peers) {
   if(!url) continue; // urlが設定されていないピアはスキップ
    pid_sockets.push([pid, url]);
}
console.log(pid_sockets);

[
  [ 'p00000000', 'https://tokyo.dncware-blockchain.com' ],
  [ 'p14048874', 'https://canada.dncware-blockchain.com' ],
  [ 'p55111259', 'https://oregon.dncware-blockchain.com' ]
]


秘密分散するシェアの数を変数Nに設定します。

In [5]:
var N = pid_sockets.length;
console.log(N);

3


ストレージコントラクトを作成します。  

In [6]:
var cid = await createObjectF('storage', 'adv13stor');

## ストレージ領域の作成
秘密分散したシェアを格納するためのストレージ領域を作成します。  

元データのファイルサイズから、必要なストレージ領域のサイズを見積もります。  

In [7]:
var filesize = 1000000; // 元データのサイズの上限
var chunksize = 40000; // ブロックチェーン環境が許容する最大サイズに設定する。通常はこのままで良い。
var chunks = 1 + Math.ceil( filesize / ( chunksize - 32*(N+1)));

ストレージコントラクトを通じて、上記パラメータでストレージ領域を作成します。  
アクセス範囲はドメイン内に限定します。  

In [8]:
var expiry = new Date('2099/12/31').valueOf();
var resp = await rpc.call(adminWallet, cid, { cmd:'create', key: 'test1', expiry, chunksize, chunks, readable:[domain], writable:[domain] });
console.log(resp);

{
  txno: 207284,
  txid: 'xy9o4MpKUdVKeKyPzR64yHEj5Hpo38QR86oizCMWwLmR9',
  status: 'ok',
  value: null
}


 作成時の引数の意味  
- key: 作成するストレージ領域を識別するための文字列  
- expiry: 有効期限(UNIXタイム)  
- chunksize: ひとつのチャンクのサイズ  
- chunks: チャンクの総数  
- readable: 読み出し権の付与先  
- writable: 書き込み権の付与先  

同様のパラメータで二つ目のストレージ領域を作成します。(上とはkeyが異なるもの)  

In [9]:
var resp = await rpc.call(adminWallet, cid, { cmd:'create', key: 'test2', expiry, chunksize, chunks, readable:[domain], writable:[domain] });
console.log(resp);

{
  txno: 207285,
  txid: 'xnKmQKGsNrsHyRyvKvGn5xGpvygieiaU5RDNYz5A8R9f4',
  status: 'ok',
  value: null
}


## 秘密分散機能のセットアップ

秘密分散のためのライブラリを設定します。

In [10]:
var T = 2; // 復元に必要なシェアの数。通常は2以上に設定する。
var sss = new api.StorageSSS(chainID, N, T, pid_sockets);

## 秘密分散機能を使ったデータの書き込み、読み出し
ファイルを秘密分散してストレージ領域へシェアを書き込みます。 

元データを用意します。

In [11]:
var file1 = api.makeRandomUint8Array(filesize);
console.log(file1);

<Buffer 7c 70 9e 2e 97 e7 ae 14 4b 82 8f fa 03 50 2d c0 39 1a 23 ee fe ca 47 c4 85 39 05 05 66 a3 d5 bd 8a 22 a5 08 f9 57 fe 92 82 fe db e7 d4 27 2c 9c 9d 75 ... 999950 more bytes>


書き込みます。

In [12]:
await sss.writeFile(adminWallet, cid, { key: 'test1' }, file1);

読み出します。

In [13]:
var file_read = await sss.readFile(adminWallet, cid, { key: 'test1' });
console.log(Buffer.from(file_read));

<Buffer 7c 70 9e 2e 97 e7 ae 14 4b 82 8f fa 03 50 2d c0 39 1a 23 ee fe ca 47 c4 85 39 05 05 66 a3 d5 bd 8a 22 a5 08 f9 57 fe 92 82 fe db e7 d4 27 2c 9c 9d 75 ... 999950 more bytes>


書き込んだファイルと読み出したファイルが一致することを確認します。

In [14]:
console.log(Buffer.compare(file_read, file1));

0


別のデータを用意します。

In [15]:
var file2 = api.makeRandomUint8Array(1000); // サイズがfilesize以下であれば、何でもよい
console.log(file2);

<Buffer 5f f7 82 85 b7 69 fd 70 5a 51 bf e8 17 53 2c a2 27 8f d7 79 84 fa fa d6 e4 1d cf 3a d2 a2 f4 2d 5b 4c c1 f6 69 90 48 6d 3c 97 88 bf 33 ca 6b 6c ce 64 ... 950 more bytes>


別のストレージ領域に書き込みます。

In [16]:
await sss.writeFile(adminWallet, cid, { key: 'test2' }, file2);

読み出します。

In [17]:
var file_read = await sss.readFile(adminWallet, cid, { key: 'test2' });
console.log(Buffer.from(file_read));

<Buffer 5f f7 82 85 b7 69 fd 70 5a 51 bf e8 17 53 2c a2 27 8f d7 79 84 fa fa d6 e4 1d cf 3a d2 a2 f4 2d 5b 4c c1 f6 69 90 48 6d 3c 97 88 bf 33 ca 6b 6c ce 64 ... 950 more bytes>


書き込んだファイルと読み出したファイルが（サイズを含めて）一致することを確認します。

In [18]:
console.log(Buffer.compare(file_read, file2));

0


ストレージ領域は独立しており、test2への書き込みは、test1に格納されたデータに影響しません。  
test1から読み出して、内容が変わっていないことを確認します。

In [19]:
var file_read = await sss.readFile(adminWallet, cid, { key: 'test1' });
console.log(Buffer.from(file_read));
console.log(Buffer.compare(file_read, file1));

<Buffer 7c 70 9e 2e 97 e7 ae 14 4b 82 8f fa 03 50 2d c0 39 1a 23 ee fe ca 47 c4 85 39 05 05 66 a3 d5 bd 8a 22 a5 08 f9 57 fe 92 82 fe db e7 d4 27 2c 9c 9d 75 ... 999950 more bytes>
0


ストレージ領域は上書き可能であり、test1へ重ねて書き込むと、前のデータは削除されます。  
test1へfile2を書き込みます。

In [20]:
await sss.writeFile(adminWallet, cid, { key: 'test1' }, file2);

test1から読み出して、内容が更新されたことを確認します。

In [21]:
var file_read = await sss.readFile(adminWallet, cid, { key: 'test1' });
console.log(Buffer.from(file_read));
console.log(Buffer.compare(file_read, file2));

<Buffer 5f f7 82 85 b7 69 fd 70 5a 51 bf e8 17 53 2c a2 27 8f d7 79 84 fa fa d6 e4 1d cf 3a d2 a2 f4 2d 5b 4c c1 f6 69 90 48 6d 3c 97 88 bf 33 ca 6b 6c ce 64 ... 950 more bytes>
0
